# Colab Mock Strategy Test

This notebook tests the LongBench strategy pipeline in Colab using the mock answer model.

What is real in this notebook:
- LongBench download/loading
- strategy execution
- compression model loading via LLMLingua-2
- summarization model loading via Hugging Face Llama
- token/context/latency table generation

What is mocked:
- answer model output
- LLM judge score
- final quality scores


## 1. Use A GPU Runtime

In Colab: `Runtime -> Change runtime type -> GPU`.

In [ ]:
from pathlib import Path
import os

print('GPU check:')
!nvidia-smi || true

## 2. Clone Repo

In [ ]:
REPO_URL = 'https://github.com/rissingh23/Context-Opt.git'
REPO_DIR = Path('/content/Context-Opt')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

## 3. Install Requirements

In [ ]:
!python3 -m pip install -r requirements.txt

## 4. Hugging Face Login

Required for the summarization strategy because it uses `meta-llama/Llama-3.1-8B-Instruct`, a gated model. The account/token must already have Llama access approved.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Verify Strategy Model Access

This checks the summarization model config before running eval. Compression will load its LLMLingua-2 model during the smoke test.

In [ ]:
from transformers import AutoConfig

summarization_model = 'meta-llama/Llama-3.1-8B-Instruct'
cfg = AutoConfig.from_pretrained(summarization_model)
print('Summarization model access OK:', cfg.model_type)

## 6. Download Small LongBench Slice

Start tiny. This downloads 5 examples per task into local JSONL files.

In [ ]:
!python3 -m src.data.download_longbench \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --limit 5

## 7. Tiny All-Strategy Smoke Test

This uses mock answering but real strategy execution. If this succeeds, all four strategy files are wired correctly.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper \
  --strategies full_context retrieval compression summarization \
  --limit 1 \
  --provider mock \
  --model mock_model \
  --summarization-model Qwen/Qwen2.5-3B-Instruct \
  --rows-output outputs/processed/mock_smoke_rows.csv \
  --aggregate-output outputs/processed/mock_smoke_summary.csv \
  --json-output outputs/processed/mock_smoke_rows.jsonl

In [ ]:
import pandas as pd
pd.read_csv('outputs/processed/mock_smoke_summary.csv')

## 8. Mock Eval On Downloaded Slice

This runs all downloaded examples. With `--limit 25` and 5 tasks x 5 examples, it covers the whole small slice.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --strategies full_context retrieval compression summarization \
  --limit 25 \
  --provider mock \
  --model mock_model \
  --summarization-model Qwen/Qwen2.5-3B-Instruct \
  --rows-output outputs/processed/mock_eval_rows.csv \
  --aggregate-output outputs/processed/mock_eval_summary.csv \
  --json-output outputs/processed/mock_eval_rows.jsonl

## 9. Render Table

In [ ]:
!python3 tools/render_eval_table.py \
  --input outputs/processed/mock_eval_summary.csv \
  --markdown-output outputs/figures/mock_eval_summary_table.md \
  --html-output outputs/figures/mock_eval_summary_table.html

In [ ]:
from IPython.display import HTML, display

display(HTML(Path('outputs/figures/mock_eval_summary_table.html').read_text()))

## 10. Inspect Errors If Any

In [ ]:
rows = pd.read_csv('outputs/processed/mock_eval_rows.csv')
errors = rows[rows['error'].fillna('') != '']
print('Error rows:', len(errors))
errors[['task', 'example_id', 'strategy', 'error']].head(20)